# 02_training.ipynb

Train the CNN classifier on encoded transaction images and compare performance with a tabular baseline.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
sys.path.append(str(Path('..').resolve()))
from src.dataset import TransactionImageDataset
from src.model import FraudCNN
from src.train import train_model, evaluate_model, save_checkpoint

data_path = Path('..') / 'data' / 'transactions_gaf_28.npz'
assert data_path.exists(), 'Run notebook 01_encoding first to generate encoded data.'
dataset = np.load(data_path)
images = dataset['images']
labels = dataset['labels']
train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=42)
train_images = images[train_idx]
train_labels = labels[train_idx]
val_images = images[val_idx]
val_labels = labels[val_idx]
train_ds = TransactionImageDataset(train_images, train_labels)
val_ds = TransactionImageDataset(val_images, val_labels)
print('Train size:', len(train_ds), 'Val size:', len(val_ds))

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FraudCNN(input_channels=1, num_classes=2).to(device)
model, history = train_model(model, train_ds, val_ds, device, batch_size=128, lr=1e-3, num_epochs=10)
save_checkpoint(model, str(Path('..') / 'outputs' / 'models' / 'fraud_cnn_gaf.pth'))

In [ ]:
val_metrics = evaluate_model(model, TransactionImageDataset(val_images, val_labels), device)
metrics_df = pd.DataFrame([val_metrics])
metrics_df[['f1', 'precision', 'recall', 'roc_auc', 'pr_auc']]

## Tabular benchmark (XGBoost)

Train a tabular fraud classifier on the original numeric features for side-by-side comparison.

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
df = pd.read_csv(Path('..') / 'data' / 'creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class'].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)
clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_val)[:, 1]
y_pred = clf.predict(X_val)
print('ROC AUC:', roc_auc_score(y_val, y_prob))
print('PR AUC:', average_precision_score(y_val, y_prob))
print(classification_report(y_val, y_pred, digits=4))